## Bentley auth and Evo Discovery using direct API calls

### Native, SPA or web app auth
#### Manual auth using common Python packages

In [ ]:
# Enter your client ID, callback URL and Evo scopes.
# NOTE: Add the scope "offline_access" to the list if you require a refresh token.

client_id = "<your-client-id>"
redirect_url = "<your-redirect-url>"
scope = ["evo.workspace", "evo.discovery", "evo.object", "evo.blocksync", "evo.file"]

base_uri = "https://ims.bentley.com"

In [ ]:
import datetime
import webbrowser
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse

import jwt
from authlib.common.security import generate_token
from authlib.integrations.requests_client import OAuth2Session


def get_access_token(
    client_id: str,
    redirect_url: str,
    authorisation_url: str,
    access_token_url: str,
    scope: str,
) -> str:
    """Get an access token."""

    port = urlparse(redirect_url).port

    class OAuthHttpServer(HTTPServer):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.authorization_code = ""

    class OAuthHttpHandler(BaseHTTPRequestHandler):
        def do_GET(self):
            self.send_response(200)
            self.send_header("Content-Type", "text/html")
            self.end_headers()
            self.wfile.write("Redirecting to the Bentley login page\n".encode("UTF-8"))

            client.fetch_token(
                url=access_token_url,
                authorization_response=self.path,
                state=state,
                code_verifier=code_verifier,
                grant_type="authorization_code",
            )

            self.wfile.write(
                """
                <html>
                  <body>
                    <h2>Authorization request to Bentley has been completed.</h1>
                    <h3>You may now close this tab or window now.</h3>
                  </body>
                </html>
                """.encode("UTF-8")
            )
            self.wfile.write(
                '<script> setTimeout("window.close()", 2500);</script>'.encode("UTF-8")
            )  # Timeout only works if already logged in

    with OAuthHttpServer(("", port), OAuthHttpHandler) as httpd:
        client = OAuth2Session(
            client_id=client_id,
            scope=scope,
            redirect_uri=redirect_url,
            code_challenge_method="S256",
        )

        code_verifier = generate_token(48)
        auth_uri, state = client.create_authorization_url(authorisation_url, code_verifier=code_verifier)
        webbrowser.open_new(auth_uri)
        httpd.handle_request()

    return client.token["access_token"]


native_access_token = get_access_token(
    client_id=client_id,
    redirect_url=redirect_url,
    authorisation_url=f"{base_uri}/connect/authorize",
    access_token_url=f"{base_uri}/connect/token",
    scope=scope,
)

print("Access token:")
print(native_access_token)

decoded = jwt.decode(native_access_token, options={"verify_signature": False}, algorithms=["RS256"])
exp_timestamp = decoded["exp"]
exp_datetime = datetime.datetime.fromtimestamp(exp_timestamp, datetime.timezone.utc)
print(f"Token expires at:\n{exp_datetime} UTC")

### Service app auth
#### Manual auth using common Python packages

In [ ]:
from http import HTTPStatus

import requests

# Enter your client ID, client secret and Evo scopes.
client_id = "<your-service-app-client-id>"
client_secret = "<your-service-app-client-secret>"

# Enter the scopes your app needs.
scope = "evo.workspace evo.discovery evo.object evo.blocksync evo.file"

# Enter the token endpoint URL.
url = "https://ims.bentley.com/connect/token"

# Enter the parameters for the token request.
params = {
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret,
    "scope": scope,
}

# Make the token request and print the access token.
response = requests.post(url, data=params)
if response.status_code != HTTPStatus.OK:
    raise RuntimeError(f"Failed to get token: {response.status_code} {response.reason}")

service_access_token = response.json()["access_token"]
print("Access token:")
print(service_access_token)

# Decode the access token to get the expiration time.
# Note: In a production application, you should verify the token signature.
decoded = jwt.decode(service_access_token, options={"verify_signature": False}, algorithms=["RS256"])
exp_timestamp = decoded["exp"]
exp_datetime = datetime.datetime.fromtimestamp(exp_timestamp, datetime.timezone.utc)
print(f"Token expires at: {exp_datetime} UTC")

### Evo Discovery API
Use an access token from either auth flow above.

In [ ]:
import json

# Choose which token to use for the Discovery API call.
access_token = native_access_token  # or service_access_token

# Prepare the authorization header.
auth_header = {"Authorization": "Bearer " + access_token}

# Define the Evo services to be searched.
services = {"blockmodel", "geoscienceobject", "file"}
params = {"service": services}

# Define the Evo Discovery API URL.
base_url = "https://discover.api.seequent.com/evo/identity/v2/discovery"

# Make the GET request to the Evo Discovery API.
response = requests.get(base_url, headers=auth_header, params=params)
response_output = json.dumps(response.json(), indent=4)
print(f"GET {base_url}?{response.url.split('?')[1]}")
print(response_output)